# 02 — Kinematics

Inspect raw SAM exports; implement reusable anatomical calculations in `src/kinematics.py` and `src/filtering.py`, **not** as untested notebook-only functions. Both modules are currently placeholders.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir() and (ROOT.parent / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise FileNotFoundError("Start Jupyter in markerless_sam/ or notebooks/")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATA = ROOT / "lunge_sam_out"
RESULTS = ROOT / "results"
print("Repository:", ROOT, "| CSV folder:", DATA.exists())

In [ ]:
SUBJECT, TRIAL, CAMERA = "S11", "lunge_01", "cam0"
RAW = DATA / SUBJECT / TRIAL / CAMERA / "sam_raw.npz"
print("SAM raw NPZ exists:", RAW.exists())

In [ ]:
if RAW.exists():
    with np.load(RAW, allow_pickle=False) as z:
        print("Available arrays:")
        for key in z.files:
            print(key, z[key].shape, z[key].dtype)
        if {"frame", "time_ms", "keypoints3d"} <= set(z.files):
            frames, t, points = z["frame"], z["time_ms"], z["keypoints3d"]
            assert len(frames) == len(t) == len(points)
            assert np.all(np.diff(frames) > 0)
            assert np.all(np.diff(t) > 0)
            print("Approximate effective sampling rate:", 1000/np.median(np.diff(t)), "Hz")
else:
    print("First transfer sam_raw.npz from the inference server.")

In [ ]:
angle_files = sorted(DATA.glob("*joint_angles.csv"))
print("Processed angle files:", len(angle_files))
for path in angle_files[:10]:
    print(path.name, list(pd.read_csv(path, nrows=0).columns))

### Implementation next
Transfer the server geometric joint-plane calculations and MHR parent-relative rotation decomposition into tested functions. Standardize hip/knee/ankle sign, coordinate frame and anatomical definitions before comparing to Vicon. Do not select Euler axes using the evaluation trials.